- docker

docker run -d --name redis -p 6379:6379 redis:7

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate

import cache

In [ ]:
class RAGState(TypedDict, total=False):
    question: str
    context: str
    sources: list
    answer: str
    cache_hit: bool

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the context below. If it's not there, say you don't know.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])
chain = prompt | llm

In [ ]:
def check_cache(state: RAGState):
    hit = cache.get_cached_answer(state["question"])
    if hit:
        return {"cache_hit": True, "answer": hit["answer"], "sources": hit["sources"]}
    return {"cache_hit": False}


def retrieve(state: RAGState):
    docs = retriever.invoke(state["question"])
    return {
        "context": "\n\n".join(d.page_content for d in docs),
        "sources": [{k: str(v) for k, v in d.metadata.items()} for d in docs],
    }


def generate(state: RAGState):
    result = chain.invoke({"context": state["context"], "question": state["question"]})
    return {"answer": result.content}


def save_to_cache(state: RAGState):
    cache.set_cached_answer(state["question"], state["answer"], state["sources"])
    return {}


In [ ]:
def route_after_cache(state: RAGState):
    return END if state["cache_hit"] else "retrieve"

In [ ]:
builder = StateGraph(RAGState)
builder.add_node("check_cache", check_cache)
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)
builder.add_node("save_to_cache", save_to_cache)

builder.add_edge(START, "check_cache")
builder.add_conditional_edges("check_cache", route_after_cache, ["retrieve", END])
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "save_to_cache")
builder.add_edge("save_to_cache", END)

graph = builder.compile()

In [ ]:
def ask(question: str) -> str:
    return graph.invoke({"question": question})["answer"]

In [ ]:
if __name__ == "__main__":
    print(ask("What is this document about?"))   # slow: retrieval + LLM
    print(ask("what is this document about?"))  